# Customer Churn Prediction — Telco

**Business problem.** A telecom operator wants to know which customers are likely to leave, so the
retention team can engage them before they go. Replacing a customer costs considerably more than
keeping one, so the deliverable is a ranked list of who to contact.

**Dataset.** IBM Telco Customer Churn — 7,043 customers, 21 columns.
**Target.** `Churn` (Yes / No).
**Model.** Decision Tree Classifier, 70:30 split, `random_state = 42`.

| Section | Contents |
|---|---|
| 0 | Setup |
| 1 | Data understanding & preparation |
| 2–7 | EDA, feature engineering, model development, evaluation, interpretation, saved model + API |

Run the cells top to bottom. The notebook reads the dataset via the relative path `../data/`, so it
must be run from inside the `notebook/` directory.

## 0. Setup

Imports and the constants used throughout. `RANDOM_STATE = 42` is mandated by the brief and is the
only seed used anywhere in the notebook.

Two library notes, because both change how the code below has to be written. **pandas 3.0** gives
text columns a dedicated `str` dtype rather than `object`, and copy-on-write is the default, so
chained assignment silently does nothing. **scikit-learn 1.9** uses `sparse_output=` on
`OneHotEncoder` (the old `sparse=` was removed) and can return pandas DataFrames from transformers
via `set_output`.

In [1]:
from io import BytesIO

import joblib
import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# The brief mandates this seed; it is the only one used in the notebook.
RANDOM_STATE = 42

DATA_PATH = "../data/TelcoCustomerChurn.csv"
DICT_PATH = "../data/TelcoCustomerChurn - Data Dictionary.csv"
TARGET = "Churn"
ID_COL = "customerID"

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

print(f"pandas {pd.__version__} | numpy {np.__version__} | scikit-learn {sklearn.__version__}")

pandas 3.0.5 | numpy 2.5.3 | scikit-learn 1.9.1


---

# 1. Data Understanding & Preparation

This section establishes what the data is, fixes what is wrong with it, and produces the train/test
split and the encoding recipe that every later section depends on.

Two constraints from the brief shape how it is ordered. **No data leakage:** nothing may be fitted
before the split, so the split at 1.10 is the boundary and the only `.fit` call in this section sits
below it. **Preprocessing must apply consistently to new data:** so the encoding is built as a
single scikit-learn object that can be pickled and reused on one unseen customer at a time, which
1.13 demonstrates.

## 1.1 Loading the data

One row is one customer at one point in time, so the prediction unit is a customer. The file is read
exactly as it sits on disk — no `na_values`, no `dtype=` overrides — because the point of the next
few cells is to *find* the defects rather than paper over them during load.

In [2]:
df = pd.read_csv(DATA_PATH)

print(f"Rows: {df.shape[0]:,}    Columns: {df.shape[1]}")
print(f"In-memory size: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df.head()

Rows: 7,043    Columns: 21
In-memory size: 7.2 MB


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 1.2 Data type and structure checks

The dtype pandas infers is a claim worth verifying, not trusting. A column of numbers stored as text
is the most common defect in a CSV and it is silent: nothing errors, the feature simply never
reaches the model.

Rather than eyeball the dtypes, the check below reads the supplied data dictionary
(`TelcoCustomerChurn - Data Dictionary.csv`) and compares the *declared* type of every column
against what pandas actually loaded.

In [3]:
df.info()

data_dict = pd.read_csv(DICT_PATH)
declared = dict(zip(data_dict["Column"], data_dict["Data Type"]))

structure = pd.DataFrame({
    "column": df.columns,
    "pandas_dtype": [str(df[c].dtype) for c in df.columns],
    "declared_type": [declared[c] for c in df.columns],
    "n_unique": [df[c].nunique() for c in df.columns],
    "example": [df[c].iloc[0] for c in df.columns],
})

# A column the dictionary calls "Numerical" should have landed as an int or a float.
declared_numeric = structure["declared_type"].str.contains("Numerical")
stored_numeric = [pd.api.types.is_numeric_dtype(df[c]) for c in df.columns]
structure["agrees"] = declared_numeric == stored_numeric

print("\n--- declared type (data dictionary) vs stored dtype (pandas) ---")
print(structure.to_string(index=False))

mismatch = structure.loc[~structure["agrees"]]
print(f"\nColumns where the two disagree: {len(mismatch)}")
print(mismatch[["column", "declared_type", "pandas_dtype"]].to_string(index=False))

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

## 1.3 Diagnosing the mis-typed column

`TotalCharges` is declared numerical but arrived as text, so something in it will not parse.
`pd.to_numeric(..., errors="coerce")` turns unparseable values into `NaN`, which locates the
offending rows without modifying anything yet.

The question that matters is not how many bad rows there are but *what kind of customer* they are,
because that is what decides whether to drop them or impute them.

In [5]:
coerced = pd.to_numeric(df["TotalCharges"], errors="coerce")
bad = coerced.isna()

print(f"Rows where TotalCharges will not parse as a number: {bad.sum()}")
print(f"Distinct raw values in those cells: {[repr(v) for v in df.loc[bad, 'TotalCharges'].unique()]}")
print(f"Their tenure values   : {sorted(df.loc[bad, 'tenure'].unique().tolist())}")
print(f"Their Churn values    : {df.loc[bad, 'Churn'].value_counts().to_dict()}")
print(f"Their Contract values : {df.loc[bad, 'Contract'].value_counts().to_dict()}")

print(f"\nRows in the WHOLE dataset with tenure == 0: {(df['tenure'] == 0).sum()}")
print("-> the unparseable rows and the zero-tenure rows are the same 11 customers.")

# Corroboration: for every billed customer, TotalCharges tracks MonthlyCharges x tenure.
billed = df["tenure"] > 0
ratio = coerced[billed] / (df.loc[billed, "MonthlyCharges"] * df.loc[billed, "tenure"])
print("\nTotalCharges / (MonthlyCharges x tenure) for customers with tenure > 0:")
print(f"  median {ratio.median():.3f} | 5th pct {ratio.quantile(0.05):.3f} | 95th pct {ratio.quantile(0.95):.3f}")
print("-> that relationship evaluated at tenure == 0 gives exactly 0.")

df.loc[bad, [ID_COL, "tenure", "MonthlyCharges", "TotalCharges", "Contract", "Churn"]]

Rows where TotalCharges will not parse as a number: 11
Distinct raw values in those cells: ["' '"]
Their tenure values   : [0]
Their Churn values    : {'No': 11}
Their Contract values : {'Two year': 10, 'One year': 1}

Rows in the WHOLE dataset with tenure == 0: 11
-> the unparseable rows and the zero-tenure rows are the same 11 customers.

TotalCharges / (MonthlyCharges x tenure) for customers with tenure > 0:
  median 1.000 | 5th pct 0.924 | 95th pct 1.075
-> that relationship evaluated at tenure == 0 gives exactly 0.


,customerID,tenure,MonthlyCharges,TotalCharges,Contract,Churn
488,4472-LVYGI,0,52.55,,Two year,No
753,3115-CZMZD,0,20.25,,Two year,No
936,5709-LVOEQ,0,80.85,,Two year,No
1082,4367-NUYAO,0,25.75,,Two year,No
1340,1371-DWPAZ,0,56.05,,Two year,No
3331,7644-OMVMY,0,19.85,,Two year,No
3826,3213-VVOLG,0,25.35,,Two year,No
4380,2520-SGTTA,0,20.00,,Two year,No
5218,2923-ARZLG,0,19.70,,One year,No
6670,4075-WKNIU,0,73.35,,Two year,No


### What this shows

`TotalCharges` is the only disagreement with the data dictionary, and the cause is 11 cells holding
a **single space**. Pandas types a column by its worst value, so one non-numeric entry demoted the
whole column to text — and a headline spend figure would have been silently skipped by any
`select_dtypes("number")` call.

All 11 rows have `tenure == 0`, and they are the *only* zero-tenure rows in the dataset. All 11 have
`Churn == "No"` and 10 of the 11 are on two-year contracts. These are brand-new customers: signed
up, monthly charge set, but no billing cycle has closed yet.

That makes the value **knowable rather than missing**. Lifetime spend for someone billed zero times
is zero — a fact the ETL wrote as blank. The ratio check corroborates it: for every billed customer
`TotalCharges` tracks `MonthlyCharges × tenure` at a median ratio of 1.000, and that relationship at
`tenure == 0` gives exactly 0. Mean-imputing instead would assign a day-one customer roughly $2,283
of history they do not have. Section 1.8 therefore fills `0.0` and keeps all 11 customers.

## 1.4 Missing-value analysis

`isna()` only recognises `NaN`, `None` and `NaT`. It does not catch a whitespace string, the text
`"unknown"`, a `-1` sentinel, or a category that means "not applicable" — so a dataset can report
zero nulls and still be full of absent information. This section therefore runs three passes rather
than one.

In [6]:
# Pass 1 - the standard check.
print("--- pass 1: df.isna().sum() ---")
print(df.isna().sum().to_string())
print(f"\nTotal recognised nulls across all {df.shape[1]} columns: {df.isna().sum().sum()}")

# Pass 2 - whitespace-only strings, which isna() cannot see.
text_cols = df.select_dtypes(include="str").columns
blank = {c: int((df[c].astype("str").str.strip() == "").sum()) for c in text_cols}
blank = {c: n for c, n in blank.items() if n > 0}
print(f"\n--- pass 2: whitespace-only cells ---")
print(blank if blank else "none")

# Pass 3 - categories that encode "not applicable".
ADDONS = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
          "TechSupport", "StreamingTV", "StreamingMovies"]

print("\n--- pass 3: sentinel levels in the service columns ---")
print(f"'No internet service' : {(df[ADDONS[0]] == 'No internet service').sum():,} rows, "
      f"in each of {len(ADDONS)} add-on columns")
print(f"'No phone service'    : {(df['MultipleLines'] == 'No phone service').sum():,} rows, "
      f"in MultipleLines only")

print("\nAre those sentinels structural, i.e. fully determined by the parent column?")
for c in ADDONS:
    same = ((df[c] == "No internet service") == (df["InternetService"] == "No")).all()
    print(f"  {c:18s} 'No internet service' <=> InternetService == 'No' : {same}")
same = ((df["MultipleLines"] == "No phone service") == (df["PhoneService"] == "No")).all()
print(f"  {'MultipleLines':18s} 'No phone service'    <=> PhoneService == 'No'    : {same}")

--- pass 1: df.isna().sum() ---
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0

Total recognised nulls across all 21 columns: 0

--- pass 2: whitespace-only cells ---
{'TotalCharges': 11}

--- pass 3: sentinel levels in the service columns ---
'No internet service' : 1,526 rows, in each of 6 add-on columns
'No phone service'    : 682 rows, in MultipleLines only

Are those sentinels structural, i.e. fully determined by the parent column?
  OnlineSecurity     'No internet service' <=> InternetService == 'No' : True
  OnlineBackup       'No internet service' <=> InternetService == 

### What this shows

The standard check reports **zero nulls in all 21 columns**, and that is misleading. The dtype is
pandas' `StringDtype`, under which `' '` is perfectly valid string content, so the 11 broken
`TotalCharges` cells are invisible to `isna()`. The whitespace sweep is what finds them.

The sentinel audit turns up two more markers, and both are **structural rather than missing**:

| Marker | Columns | Rows | Determined by |
|---|---|---|---|
| `No internet service` | 6 add-on columns | 1,526 each | `InternetService == "No"` |
| `No phone service` | `MultipleLines` | 682 | `PhoneService == "No"` |

The checks above confirm each marker appears *if and only if* the parent column says so —
exhaustively, with no exceptions. So there is no missingness mechanism to model here: the value is a
deterministic function of another column and records a real fact about the customer ("has no online
backup, because they have no internet"). Both are kept as their own level. Collapsing them into
`"No"` would merge two genuinely different groups — customers who declined an add-on, and customers
who could not buy one — and section 2 will show those groups behave differently.

## 1.5 Duplicate analysis

Three different questions, and the second matters most. Fully identical rows are a data-quality
fault. A repeated **identifier** is worse: the same customer could land in both the train and test
sets, which inflates the test score for no real reason. Rows that match once the identifier is
ignored are a third case, and one that needs interpreting rather than fixing.

In [7]:
print(f"Fully duplicated rows               : {df.duplicated().sum()}")
print(f"Duplicated customerID values        : {df[ID_COL].duplicated().sum()}")
print(f"Unique customerIDs                  : {df[ID_COL].nunique():,} of {len(df):,} rows")
print(f"Duplicated rows ignoring customerID : {df.drop(columns=[ID_COL]).duplicated().sum()}")

print("\nThe last figure counts distinct customers who happen to share an attribute profile")
print("(e.g. short-tenure, phone-only, month-to-month). That is expected in a 19-field")
print("categorical space, is not a data fault, and carries no train/test leakage risk because")
print("each is a genuinely separate customer. Nothing is dropped.")

Fully duplicated rows               : 0
Duplicated customerID values        : 0
Unique customerIDs                  : 7,043 of 7,043 rows
Duplicated rows ignoring customerID : 22

The last figure counts distinct customers who happen to share an attribute profile
(e.g. short-tenure, phone-only, month-to-month). That is expected in a 19-field
categorical space, is not a data fault, and carries no train/test leakage risk because
each is a genuinely separate customer. Nothing is dropped.


## 1.6 Numerical and categorical feature identification

These lists drive the encoder, so getting them right prevents a whole class of downstream bug. It is
a modelling decision rather than a dtype lookup, because dtype misleads in both directions here:
`TotalCharges` is numeric but stored as text, and `SeniorCitizen` is stored as `int64` but is really
a binary flag — there is no meaningful "1.5 senior citizens".

`SeniorCitizen` therefore gets its own list. One-hot encoding it would emit a perfectly collinear
pair of columns carrying a single bit, and passthrough is also the safer route at inference time: a
`OneHotEncoder` fitted on `int64` silently encodes the string `"1"` as *neither* level, whereas a
passthrough column either works or fails loudly on dtype.

`customerID` is excluded from the model entirely. It is unique per row, so a tree given it could
split every customer into their own leaf and score near-perfectly on train — and it carries no
transferable signal, because every customer arriving in production has an ID the model has never
seen.

In [9]:
numeric_features = ["tenure", "MonthlyCharges", "TotalCharges"]
binary_features = ["SeniorCitizen"]
categorical_features = [
    c for c in df.columns
    if c not in numeric_features + binary_features + [TARGET, ID_COL]
]

# Coverage and disjointness. These asserts catch a typo here rather than as a
# KeyError halfway through section 4.
assert (set(numeric_features) | set(binary_features) | set(categorical_features)
        | {TARGET, ID_COL}) == set(df.columns), "feature lists do not cover every column"
assert not set(numeric_features) & set(binary_features)
assert not set(numeric_features) & set(categorical_features)
assert not set(binary_features) & set(categorical_features)
assert len(categorical_features) == 15

print(f"numeric_features     ({len(numeric_features)}): {numeric_features}")
print(f"binary_features      ({len(binary_features)}): {binary_features}")
print(f"categorical_features ({len(categorical_features)}): {categorical_features}")
print(f"target: {TARGET!r}  |  excluded identifier: {ID_COL!r}")
print(f"\nCoverage: {len(numeric_features)} + {len(binary_features)} + "
      f"{len(categorical_features)} + target + id = {df.shape[1]} columns. Asserts passed.")

print("\nWhy these are hand-declared rather than inferred:")
print(f"  select_dtypes('number') returns {df.select_dtypes(include='number').columns.tolist()}")
print("  -> wrong on both ends: it picks up SeniorCitizen (a binary flag) and")
print("     misses TotalCharges (still text at this point in the notebook).")

print("\n--- cardinality of the categorical features ---")
print(df[categorical_features].nunique().sort_values().to_string())

numeric_features     (3): ['tenure', 'MonthlyCharges', 'TotalCharges']
binary_features      (1): ['SeniorCitizen']
categorical_features (15): ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
target: 'Churn'  |  excluded identifier: 'customerID'

Coverage: 3 + 1 + 15 + target + id = 21 columns. Asserts passed.

Why these are hand-declared rather than inferred:
  select_dtypes('number') returns ['SeniorCitizen', 'tenure', 'MonthlyCharges']
  -> wrong on both ends: it picks up SeniorCitizen (a binary flag) and
     misses TotalCharges (still text at this point in the notebook).

--- cardinality of the categorical features ---
gender              2
Partner             2
Dependents          2
PhoneService        2
PaperlessBilling    2
MultipleLines       3
OnlineSecurity      3
InternetService     3
On

## 1.7 Target-variable analysis

The class balance decides which metrics mean anything. If most customers do not churn, a model that
predicts "No" for everyone scores well on accuracy while being useless to the retention team — and
that number is the baseline every later result has to be judged against.

Numbers only here. The churn-distribution chart belongs to section 2, where the brief asks for it as
one of the required visualisations.

In [10]:
counts = df[TARGET].value_counts()
pct = df[TARGET].value_counts(normalize=True).reindex(counts.index) * 100

print(pd.DataFrame({"count": counts, "percent": pct.round(3)}).to_string())

baseline = counts.max() / len(df)
print(f"\nChurn rate                       : {pct['Yes']:.2f}%")
print(f"Class ratio (No : Yes)           : {counts['No'] / counts['Yes']:.2f} : 1")
print(f"Majority-class baseline accuracy : {baseline:.2%}   <-- the number section 5 must beat")
print(f"\nA model predicting 'No' for all {len(df):,} customers scores {baseline:.2%} accuracy")
print(f"and identifies 0 of the {counts['Yes']:,} churners. Accuracy alone cannot detect that.")

       count  percent
Churn                
No      5174   73.463
Yes     1869   26.537

Churn rate                       : 26.54%
Class ratio (No : Yes)           : 2.77 : 1
Majority-class baseline accuracy : 73.46%   <-- the number section 5 must beat

A model predicting 'No' for all 7,043 customers scores 73.46% accuracy
and identifies 0 of the 1,869 churners. Accuracy alone cannot detect that.


### What this shows

1,869 of 7,043 customers churned — **26.54%** — so the majority-class baseline is **73.46%
accuracy**. Any accuracy figure in section 5 that is not clearly above 73.46% means the model has
learned nothing.

At roughly 2.8:1 the imbalance is mild. For comparison, credit-card fraud runs nearer 580:1, where
resampling or an anomaly-detection framing is needed just to get a model that predicts the positive
class at all. Here one training row in four is a churner, which is ample signal for a decision tree,
so no resampling is required.

What the imbalance does change is the **choice of metric**: accuracy is flattered by the 73.46%
floor, so section 5 leads on recall and precision for the churn class rather than on accuracy.
`class_weight="balanced"` is still worth testing in section 4 as one of the compared configurations.

## 1.8 Data cleaning

Acting on what the previous cells found. The cell works on a copy so `df` stays pristine and the cell
is safe to re-run.

| Decision | Why |
|---|---|
| `TotalCharges` → numeric | The data dictionary declares it numerical; a single space made pandas type the whole column as text. |
| The 11 `NaN` → `0.0` | `tenure == 0` means no billing cycle has closed, so lifetime spend is a *known* 0. Mean imputation would invent ≈ $2,283 for a never-billed customer. |
| Drop `customerID` | Unique per row: memorisable by a tree, and meaningless for a customer the model has never seen. |
| Keep all 7,043 rows | No duplicates and no true missingness. Dropping the 11 would delete the entire new-joiner segment, which is exactly the cohort retention cares about. |
| Keep the service sentinels as levels | Proved structural at 1.4 — they record which product a customer holds, not a gap. |
| No outlier treatment | The `MonthlyCharges` / `TotalCharges` ranges are the product's own price grid, not error. Trees split on rank order, so extremes cannot dominate a split the way they would in a linear model. |

In [11]:
df_clean = df.copy()   # df stays pristine, so this cell is idempotent on re-run

# 1. Mis-typed column -> proper numeric dtype. The 11 ' ' cells become NaN on coercion.
df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce")

# 2. Those 11 never-billed customers have a known lifetime spend of 0.
n_filled = int(df_clean["TotalCharges"].isna().sum())
df_clean["TotalCharges"] = df_clean["TotalCharges"].fillna(0.0)

# 3. Drop the identifier.
df_clean = df_clean.drop(columns=[ID_COL])

print(f"TotalCharges dtype : {df['TotalCharges'].dtype} -> {df_clean['TotalCharges'].dtype}")
print(f"Imputed 0.0 for    : {n_filled} never-billed (tenure == 0) customers")
print(f"Dropped column     : {ID_COL!r}")
print(f"Shape              : {df.shape} -> {df_clean.shape}   (all rows kept, 1 column removed)")

# --- checks ---
assert df_clean.isna().sum().sum() == 0, "nulls remain after cleaning"
assert len(df_clean) == len(df) == 7043, "rows were lost"
assert df_clean["TotalCharges"].dtype.kind == "f", "TotalCharges is not float"
assert ID_COL not in df_clean.columns, "identifier still present"
assert (df_clean.loc[df_clean["tenure"] == 0, "TotalCharges"] == 0).all()
residual = {c: int((df_clean[c].astype("str").str.strip() == "").sum())
            for c in df_clean.select_dtypes(include="str").columns}
assert sum(residual.values()) == 0, f"whitespace-only cells remain: {residual}"
print("\nCleaning checks passed.")

print("\n--- numeric summary after cleaning ---")
print(df_clean[numeric_features].describe().T.round(2).to_string())

TotalCharges dtype : str -> float64
Imputed 0.0 for    : 11 never-billed (tenure == 0) customers
Dropped column     : 'customerID'
Shape              : (7043, 21) -> (7043, 20)   (all rows kept, 1 column removed)

Cleaning checks passed.

--- numeric summary after cleaning ---
                 count     mean      std    min     25%      50%      75%      max
tenure          7043.0    32.37    24.56   0.00    9.00    29.00    55.00    72.00
MonthlyCharges  7043.0    64.76    30.09  18.25   35.50    70.35    89.85   118.75
TotalCharges    7043.0  2279.73  2266.79   0.00  398.55  1394.55  3786.60  8684.80


## 1.9 Features, target, and the positive class

Separating `X` from `y`, and encoding the target to 0/1. Which class becomes `1` matters: precision
and recall are defined relative to the positive class, so mapping it backwards would invert every
metric in section 5. The convention is the event being detected, so `Yes` (churned) → `1`. The
crosstab below is printed as visible proof of the mapping rather than a claim about it.

In [12]:
X = df_clean.drop(columns=[TARGET])
y = (df_clean[TARGET] == "Yes").astype(int)

print("--- target encoding check ---")
print(pd.crosstab(df_clean[TARGET], y, rownames=["Churn"], colnames=["y"]).to_string())

assert y[df_clean[TARGET] == "Yes"].eq(1).all(), "positive class is not 'Yes'"
assert y[df_clean[TARGET] == "No"].eq(0).all(), "negative class is not 'No'"
assert y.sum() == 1869
assert y.dtype.kind == "i"

print(f"\nX: {X.shape}    y: {y.shape}    'Yes' -> 1  ({y.sum():,} churners)")
print("Precision and recall in section 5 therefore refer to churners.")

--- target encoding check ---
y         0     1
Churn            
No     5174     0
Yes       0  1869

X: (7043, 19)    y: (7043,)    'Yes' -> 1  (1,869 churners)
Precision and recall in section 5 therefore refer to churners.


## 1.10 Train/test split — the leakage boundary

70:30, `random_state = 42`, stratified on the target. This cell is the line the brief's leakage
warning is about, so it is worth being explicit about what is being protected.

The test set exists to stand in for data that does not exist yet, and that job requires the model to
be genuinely ignorant of it. If a scaler's mean or an encoder's category list is learned from all
7,043 rows and then used in training, every training row encodes a fact about the test set; the test
score stops estimating production performance and quietly becomes optimistic. Nothing errors — the
numbers are just slightly too good, uniformly, in a way no amount of staring at the notebook
reveals. **Everything below this cell that learns, learns from `X_train` only**, and the single
`.fit` call in this section is at 1.12.

`stratify=y` is not cosmetic either. With a 26.5% positive class an unlucky random split can shift
the test churn rate by several points, which moves the very baseline the model is measured against.
The cell quantifies that rather than asserting it.

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)

print(f"X_train {X_train.shape}   y_train {y_train.shape}")
print(f"X_test  {X_test.shape}   y_test  {y_test.shape}")
print(f"Split ratio: {len(X_train) / len(X):.1%} / {len(X_test) / len(X):.1%}")

print("\n--- churn rate must match across all three ---")
print(f"full  : {y.mean():.4f}   ({y.sum():,} of {len(y):,})")
print(f"train : {y_train.mean():.4f}   ({y_train.sum():,} of {len(y_train):,})")
print(f"test  : {y_test.mean():.4f}   ({y_test.sum():,} of {len(y_test):,})")
print(f"train-vs-test gap: {abs(y_train.mean() - y_test.mean()):.4f}")

# --- checks ---
assert X_train.shape == (4930, 19) and X_test.shape == (2113, 19)
assert len(X_train) + len(X_test) == 7043
assert abs(y_train.mean() - y_test.mean()) < 0.005, "stratification failed"
assert X_train.index.intersection(X_test.index).empty, "a customer appears in both splits"
assert list(X_train.columns) == list(X_test.columns)
print("\nSplit checks passed, including no index overlap between train and test.")

# Evidence for stratify=y, rather than an assertion about it.
rates = [train_test_split(X, y, test_size=0.30, random_state=s)[3].mean() for s in range(200)]
print(f"\nUnstratified test churn rate across 200 seeds: {min(rates):.2%} to {max(rates):.2%} "
      f"(spread {max(rates) - min(rates):.2%})")
print(f"Stratified: {y_test.mean():.2%}, on every seed.")

X_train (4930, 19)   y_train (4930,)
X_test  (2113, 19)   y_test  (2113,)
Split ratio: 70.0% / 30.0%

--- churn rate must match across all three ---
full  : 0.2654   (1,869 of 7,043)
train : 0.2653   (1,308 of 4,930)
test  : 0.2655   (561 of 2,113)
train-vs-test gap: 0.0002

Split checks passed, including no index overlap between train and test.

Unstratified test churn rate across 200 seeds: 24.18% to 29.11% (spread 4.92%)
Stratified: 26.55%, on every seed.


### What this shows

4,930 training and 2,113 test customers, with churn rates of 26.53% and 26.55% — a gap of 0.0002.
The index-intersection check confirms no customer appears in both halves.

The stratification evidence is the striking part: across 200 unstratified seeds the test churn rate
ranges from **24.18% to 29.11%**. That is a five-point swing in the quantity being predicted, caused
by nothing but seed luck, and it moves the majority-class baseline between roughly 70.9% and 75.8%.
A model scoring 74% would be beating the baseline or losing to it depending on which split happened
to be drawn. Because the two halves are complementary, a test set light on churners also leaves a
training set heavy with them, so the model learns one prior and is scored against another.

`stratify=y` removes that variance by sampling within each class, so the test churn rate is 26.55%
on every seed and model comparisons in sections 4–5 differ because the *models* differ.

## 1.11 Encoding architecture

The brief asks for encoding of categorical variables in this section *and* for no data leakage. Both
are satisfied by separating the recipe from the fit: this cell **defines** the transformation and
learns nothing from any data, so it cannot leak. 1.12 then fits it on the training set alone.

`build_preprocessor()` is a factory rather than one shared object because `Pipeline` fits its steps
in place without cloning them — handing the same instance to two pipelines would let the second
silently overwrite the first one's fitted state. Sections 3 and 4 call the factory for a fresh
transformer; the fitted object from 1.12 is kept only for inspection.

The choices inside it, and why each one:

- **`handle_unknown="ignore"`** — a category the model has never seen must not crash the section 7
  API. An unknown value becomes an all-zero block instead.
- **`drop=None`, not `drop="first"`** — those two options interact badly. With
  `handle_unknown="ignore"`, an unknown value and the dropped reference level *both* encode to all
  zeros, so an unseen contract type would be silently relabelled as `Month-to-month`. Keeping every
  level makes unknowns distinguishable. Trees are indifferent to the dummy-variable trap, so the
  usual reason to drop a level does not apply here.
- **`SeniorCitizen` passed through** rather than one-hot encoded, per 1.6.
- **A constant-`0.0` imputer on the numeric branch** — 1.8 repaired `TotalCharges` in pandas, where
  the reasoning could be explained, but that repair lives in a notebook cell and not in the saved
  artifact, so a brand-new customer arriving at the API would not get it. This closes the gap inside
  the picklable object. On training data it is a no-op, because the values are already filled.
- **No scaler** — a decision tree splits on rank order within a single feature, so any monotone
  rescaling is invisible to it and `StandardScaler` here produces an identical tree. The
  `scale_numeric` switch exists and is off by default, so the omission is a decision rather than an
  oversight, and a bonus logistic-regression comparison in section 4 can turn it on without a
  rewrite.
- **`set_output("pandas")` with `verbose_feature_names_out=False`** — keeps names readable
  (`Contract_Two year`, not `cat__Contract_Two year`) and gives the fitted model a real
  `feature_names_in_`, which is what makes section 6's tree plot and importance table legible.

In [14]:
def build_preprocessor(numeric=None, binary=None, categorical=None, scale_numeric=False):
    """Return a fresh, UNFITTED ColumnTransformer for the churn features.

    A factory rather than a module-level object: Pipeline fits its steps in place
    without cloning them, so sharing one instance across pipelines would let the
    second silently overwrite the first one's fitted state.

    Sections 3 and 4 call this. Section 3 can pass extended feature lists to make
    room for engineered columns without editing section 1.
    """
    numeric = numeric_features if numeric is None else list(numeric)
    binary = binary_features if binary is None else list(binary)
    categorical = categorical_features if categorical is None else list(categorical)

    # Constant-0.0 imputation rather than passthrough, so a never-billed customer
    # arriving at the section 7 API is repaired inside the pickled artifact exactly
    # as section 1.8 repaired the training data in pandas.
    numeric_branch = SimpleImputer(strategy="constant", fill_value=0.0)
    if scale_numeric:
        numeric_branch = Pipeline([("impute", numeric_branch), ("scale", StandardScaler())])

    return ColumnTransformer(
        transformers=[
            ("num", numeric_branch, list(numeric)),
            ("flag", "passthrough", list(binary)),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop=None),
             list(categorical)),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")


print("build_preprocessor() defined. This cell fits nothing and learns nothing,")
print("so no information can cross the train/test boundary here.")
build_preprocessor()

build_preprocessor() defined. This cell fits nothing and learns nothing,
so no information can cross the train/test boundary here.


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('flag', ...), ...]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transforme

## 1.12 Fitting the encoder on the training set

The encoder's category vocabulary is learned from the 4,930 training rows and then held fixed; the
2,113 test rows are passed through that fixed mapping. This is the cell that discharges the brief's
"encoding of categorical variables" requirement, and it sits below the split so the leakage boundary
is visible in cell order.

The expected width is `3 numeric + 1 binary flag + 41 one-hot = 45`, and the cell checks that
arithmetic against the encoder's own output rather than trusting it. The final check — that every
one-hot block sums to exactly 1 on every training row — is a single strong test that the encoder
covered every level of every categorical column.

In [15]:
preprocessor = build_preprocessor().fit(X_train)     # the only .fit in section 1

X_train_encoded = preprocessor.transform(X_train)
X_test_encoded = preprocessor.transform(X_test)
encoded_feature_names = list(preprocessor.get_feature_names_out())

cat_levels = preprocessor.named_transformers_["cat"].categories_
n_onehot = sum(len(c) for c in cat_levels)
expected = len(numeric_features) + len(binary_features) + n_onehot

print(f"Features: {X_train.shape[1]} raw -> {X_train_encoded.shape[1]} encoded")
print(f"  {len(numeric_features)} numeric + {len(binary_features)} binary flag + "
      f"{n_onehot} one-hot = {expected}")
print(f"X_train_encoded {X_train_encoded.shape}    X_test_encoded {X_test_encoded.shape}")

print("\n--- one-hot expansion per categorical feature ---")
for name, levels in zip(categorical_features, cat_levels):
    print(f"  {name:18s} {len(levels)} -> {list(levels)}")

# --- checks ---
assert X_train_encoded.shape[1] == 45 == expected, "unexpected encoded width"
assert X_train_encoded.shape[1] == X_test_encoded.shape[1]
assert list(X_train_encoded.columns) == list(X_test_encoded.columns)
assert not X_train_encoded.columns.duplicated().any(), "duplicate encoded feature names"
assert X_train_encoded.notna().all().all()
assert preprocessor.n_features_in_ == 19
for name in categorical_features:
    block = X_train_encoded.filter(regex=f"^{name}_")
    assert block.sum(axis=1).eq(1).all(), f"{name} one-hot block does not sum to 1"
print("\nEncoding checks passed. Every one-hot block sums to exactly 1 per training row.")

print(f"\nThe {len(encoded_feature_names)} encoded feature names:")
print(encoded_feature_names)
X_train_encoded.head(3)

Features: 19 raw -> 45 encoded
  3 numeric + 1 binary flag + 41 one-hot = 45
X_train_encoded (4930, 45)    X_test_encoded (2113, 45)

--- one-hot expansion per categorical feature ---
  gender             2 -> ['Female', 'Male']
  Partner            2 -> ['No', 'Yes']
  Dependents         2 -> ['No', 'Yes']
  PhoneService       2 -> ['No', 'Yes']
  MultipleLines      3 -> ['No', 'No phone service', 'Yes']
  InternetService    3 -> ['DSL', 'Fiber optic', 'No']
  OnlineSecurity     3 -> ['No', 'No internet service', 'Yes']
  OnlineBackup       3 -> ['No', 'No internet service', 'Yes']
  DeviceProtection   3 -> ['No', 'No internet service', 'Yes']
  TechSupport        3 -> ['No', 'No internet service', 'Yes']
  StreamingTV        3 -> ['No', 'No internet service', 'Yes']
  StreamingMovies    3 -> ['No', 'No internet service', 'Yes']
  Contract           3 -> ['Month-to-month', 'One year', 'Two year']
  PaperlessBilling   2 -> ['No', 'Yes']
  PaymentMethod      4 -> ['Bank transfer (automa

,tenure,MonthlyCharges,TotalCharges,SeniorCitizen,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,PhoneService_No,PhoneService_Yes,MultipleLines_No,MultipleLines_No phone service,MultipleLines_Yes,InternetService_DSL,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,OnlineBackup_No,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No,DeviceProtection_No internet service,DeviceProtection_Yes,TechSupport_No,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
5557,5.0,80.20,384.25,0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2270,3.0,86.85,220.95,1,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
6930,3.0,75.15,216.75,0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0


## 1.13 Applying the same transformation to unseen data

The brief's closing requirement is that preprocessing "can also be applied consistently to
new/unseen data". That is easy to claim and worth demonstrating, because the section 7 API will
receive exactly one customer at a time — a very different shape from the 4,930-row frame the encoder
was fitted on.

Five checks: a single-row frame transforms to the same 45 columns; column order does not matter,
because the `ColumnTransformer` selects by name; an unseen category degrades to an all-zero block
instead of raising; a *missing* column fails loudly rather than silently; and the fitted object
survives a pickle round-trip, which is what section 7 actually depends on.

The resulting API contract is two lines: build a one-row `DataFrame` keyed by column name, and coerce
the numeric fields with `pd.to_numeric(..., errors="coerce")` — the imputer in the pipeline handles
the rest.

In [16]:
# A single new customer, exactly the payload shape section 7's API will receive.
new_customer = {
    "gender": "Female", "SeniorCitizen": 0, "Partner": "Yes", "Dependents": "No",
    "tenure": 3, "PhoneService": "Yes", "MultipleLines": "No",
    "InternetService": "Fiber optic", "OnlineSecurity": "No", "OnlineBackup": "No",
    "DeviceProtection": "No", "TechSupport": "No", "StreamingTV": "Yes",
    "StreamingMovies": "Yes", "Contract": "Month-to-month", "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check", "MonthlyCharges": 94.35, "TotalCharges": 283.05,
}
one_row = pd.DataFrame([new_customer])
encoded_one = preprocessor.transform(one_row)
print(f"1. One-row frame -> {encoded_one.shape}. Same 45 columns as training.")

# 2. Column order must not matter: the ColumnTransformer selects by name.
reversed_cols = one_row[one_row.columns[::-1]]
assert np.allclose(preprocessor.transform(reversed_cols).to_numpy(), encoded_one.to_numpy())
print("2. Columns reversed -> bit-identical output (selection is by name, not position).")

# 3. An unseen category must not crash the API.
unseen = one_row.copy()
unseen["Contract"] = "Three year"
contract_block = preprocessor.transform(unseen).filter(regex="^Contract_")
assert contract_block.to_numpy().sum() == 0
print(f"3. Unseen Contract='Three year' -> Contract block {contract_block.to_numpy().tolist()}, "
      "no exception.")
print("   Distinguishable from every real level, which is why drop=None was chosen.")

# 4. A missing column must fail loudly, not silently.
try:
    preprocessor.transform(one_row.drop(columns=["Contract"]))
    raise AssertionError("a missing column should have raised ValueError")
except ValueError as err:
    print(f"4. Missing column -> ValueError: {str(err).splitlines()[0]}")

# 5. The fitted object must survive pickling, or section 7 cannot load it.
buf = BytesIO()
joblib.dump(preprocessor, buf)
buf.seek(0)
assert np.allclose(joblib.load(buf).transform(one_row).to_numpy(), encoded_one.to_numpy())
print(f"5. joblib round-trip ({buf.getbuffer().nbytes / 1024:.1f} KB) -> identical output.")

encoded_one

1. One-row frame -> (1, 45). Same 45 columns as training.
2. Columns reversed -> bit-identical output (selection is by name, not position).
3. Unseen Contract='Three year' -> Contract block [[0.0, 0.0, 0.0]], no exception.
   Distinguishable from every real level, which is why drop=None was chosen.
4. Missing column -> ValueError: columns are missing: {'Contract'}
5. joblib round-trip (8.1 KB) -> identical output.


,tenure,MonthlyCharges,TotalCharges,SeniorCitizen,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,PhoneService_No,PhoneService_Yes,MultipleLines_No,MultipleLines_No phone service,MultipleLines_Yes,InternetService_DSL,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,OnlineBackup_No,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No,DeviceProtection_No internet service,DeviceProtection_Yes,TechSupport_No,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,3.0,94.35,283.05,0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


## 1.14 Summary and hand-off

**What the data turned out to be.** 7,043 customers, 21 columns, one row per customer. The only
structural defect was `TotalCharges` arriving as text, because 11 never-billed customers had a blank
cell; those were imputed to a known `0.0` rather than dropped. `isna()` reported a clean dataset and
was wrong. Two sentinel levels (`No internet service`, `No phone service`) proved to be structural
facts rather than missingness and were kept as their own categories. No duplicate rows and no
duplicate identifiers. The target is 26.54% positive, giving a **73.46% majority-class baseline**.

**How leakage is prevented.** The split at 1.10 is the boundary. The only `.fit` in this section is
at 1.12 and takes `X_train`. The test set is touched only by `transform` and by shape reporting, and
stays sealed until section 5.

**Requirements traceability.**

| Brief requirement | Where |
|---|---|
| Data type and structure checks | 1.2 |
| Missing-value analysis | 1.3, 1.4 |
| Duplicate analysis | 1.5 |
| Numerical and categorical feature identification | 1.6 |
| Target-variable analysis | 1.7 |
| Data cleaning and preprocessing | 1.8 |
| Encoding of categorical variables | 1.11, 1.12 |
| Train/test split, 70:30 | 1.10 |
| `random_state = 42` | 0 (constant), 1.10 |
| Avoid data leakage | 1.10, 1.12 |
| Preprocessing applies to new/unseen data | 1.13 |

**Contract for later sections.** Section 2 should build its own
`train_df = X_train.join(y_train.rename(TARGET))` and read only that, so no visual insight can leak
a test-set fact into a modelling decision. Section 3 should append engineered column names to a copy
of the feature lists and call `build_preprocessor(numeric=..., categorical=...)` rather than editing
this section — and must compute engineered features from within-row information only, never from a
training-set aggregate, or leakage re-enters through the back door. Section 4 must call
`build_preprocessor()` for a fresh transformer and train on raw `X_train` through a `Pipeline`, not
on `X_train_encoded`.

In [17]:
checks = {
    "feature lists cover every column": (
        (set(numeric_features) | set(binary_features) | set(categorical_features)
         | {TARGET, ID_COL}) == set(df.columns)),
    "no nulls after cleaning": df_clean.isna().sum().sum() == 0,
    "all 7,043 rows retained": len(df_clean) == 7043,
    "positive class 'Yes' -> 1": bool(y[df_clean[TARGET] == "Yes"].eq(1).all()),
    "70:30 split shapes correct": X_train.shape == (4930, 19) and X_test.shape == (2113, 19),
    "no customer in both splits": X_train.index.intersection(X_test.index).empty,
    "stratified within 0.5pp": abs(y_train.mean() - y_test.mean()) < 0.005,
    "encoder saw 19 raw features": preprocessor.n_features_in_ == 19,
    "45 encoded features, train == test": (
        X_train_encoded.shape[1] == X_test_encoded.shape[1] == 45),
}
for name, ok in checks.items():
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
assert all(checks.values()), "a section 1 check failed"
print(f"\nAll {len(checks)} section 1 checks passed.\n")

manifest = [
    ("df", str(df.shape), "raw, never mutated"),
    ("df_clean", str(df_clean.shape), "TotalCharges numeric, customerID dropped"),
    ("X / y", f"{X.shape} / {y.shape}", "model input; target as 0/1"),
    ("X_train / y_train", f"{X_train.shape} / {y_train.shape}",
     "70% - everything downstream learns from this"),
    ("X_test / y_test", f"{X_test.shape} / {y_test.shape}", "30% - sealed until section 5"),
    ("numeric_features", str(len(numeric_features)), str(numeric_features)),
    ("binary_features", str(len(binary_features)), str(binary_features)),
    ("categorical_features", str(len(categorical_features)), "one-hot encoded"),
    ("build_preprocessor", "factory", "call for a fresh unfitted transformer (sections 3-4)"),
    ("preprocessor", "fitted", "fitted on X_train only; inspection + section 6 naming"),
    ("encoded_feature_names", str(len(encoded_feature_names)), "names of the encoded columns"),
    ("X_train_encoded / X_test_encoded", f"{X_train_encoded.shape} / {X_test_encoded.shape}",
     "INSPECTION ONLY - models consume the pipeline"),
]
print("--- exported for sections 2-7 ---")
width = max(len(n) for n, _, _ in manifest)
for name, shape, note in manifest:
    print(f"  {name:<{width}}  {shape:<24}  {note}")

  [PASS] feature lists cover every column
  [PASS] no nulls after cleaning
  [PASS] all 7,043 rows retained
  [PASS] positive class 'Yes' -> 1
  [PASS] 70:30 split shapes correct
  [PASS] no customer in both splits
  [PASS] stratified within 0.5pp
  [PASS] encoder saw 19 raw features
  [PASS] 45 encoded features, train == test

All 9 section 1 checks passed.

--- exported for sections 2-7 ---
  df                                (7043, 21)                raw, never mutated
  df_clean                          (7043, 20)                TotalCharges numeric, customerID dropped
  X / y                             (7043, 19) / (7043,)      model input; target as 0/1
  X_train / y_train                 (4930, 19) / (4930,)      70% - everything downstream learns from this
  X_test / y_test                   (2113, 19) / (2113,)      30% - sealed until section 5
  numeric_features                  3                         ['tenure', 'MonthlyCharges', 'TotalCharges']
  binary_features         